In [ ]:
from importlib import import_module
from pathlib import Path
import json
import random
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / '.data/extracted').is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Open this notebook from inside the project.')
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

evaluation = import_module('src.steps.04_evaluation')
EvaluationItem = evaluation.EvaluationItem
Evaluator = evaluation.Evaluator
hf_inference = evaluation.hf_inference
ollama_inference = evaluation.ollama_inference
openai_inference = evaluation.openai_inference
render_json_prompt = evaluation.render_json_prompt
report_results = evaluation.report_results

SAMPLE_SIZE, SEED, WORKERS = None, 42, 1


In [ ]:
def load_items(limit=None, seed=42):
    test_file = PROJECT_ROOT / '.data/curated/test.jsonl'
    if not test_file.is_file():
        raise FileNotFoundError('Run the data-curation notebook first to create .data/curated/test.jsonl.')
    records = [json.loads(line) for line in test_file.read_text(encoding='utf-8').splitlines() if line.strip()]
    available = len(records)
    if limit is not None:
        if limit < 1:
            raise ValueError('limit must be positive or None.')
        records = random.Random(seed).sample(records, min(limit, available))

    items = []
    for record in records:
        messages = {message['role']: message['content'] for message in record['messages']}
        items.append(EvaluationItem(
            resume_id=record['resume_id'],
            category=record['category'],
            resume_text=messages['user'],
            target_json=json.loads(messages['assistant']),
        ))
    return items, available

In [ ]:
items, available_pairs = load_items(SAMPLE_SIZE, SEED)
print(f'Available held-out test resumes: {available_pairs:,}')
print(f'Loaded for this test: {len(items)}')

In [ ]:
evaluator = Evaluator(items)

In [ ]:
print(render_json_prompt(items[0].resume_text))

In [ ]:
OLLAMA_MODEL = 'llama3.2'


In [ ]:
callback = ollama_inference(OLLAMA_MODEL)


In [ ]:
callback(items[0])


In [ ]:

# results = evaluator.evaluate(
#     callback=ollama_inference(OLLAMA_MODEL),
#     workers=WORKERS,
# )

In [ ]:
# for result in results.results:
#     print({
#         'resume_id': result.item.resume_id,
#         'overall': None if result.overall_score is None else round(result.overall_score * 100, 2),
#         'programmatic': None if result.programmatic_score is None else round(result.programmatic_score * 100, 2),
#         'semantic': None if result.semantic_score is None else round(result.semantic_score * 100, 2),
#         'identity': None if result.identity_score is None else round(result.identity_score * 100, 2),
#     })

# evaluation.report_results(results)


In [ ]:
OLLAMA_MODELS = ['llama3.2', 'gemma3:270m']

reports = {}
for model_name in OLLAMA_MODELS:
    reports[f'ollama:{model_name}'] = evaluator.evaluate(
        callback=ollama_inference(model_name),
        workers=WORKERS,
    )


In [ ]:
OPENROUTER_MODELS = [
    'meta-llama/llama-3.2-1b-instruct',
]

for model_name in OPENROUTER_MODELS:
    reports[f'openrouter:{model_name}'] = evaluator.evaluate(
        callback=openai_inference(model_name),
        workers=WORKERS,
    )



In [ ]:
HUGGINGFACE_MODELS = [
    'google/gemma-3-270m-it',
    'HuggingFaceTB/SmolLM2-360M-Instruct',
]

for model_name in HUGGINGFACE_MODELS:
    reports[f'huggingface:{model_name}'] = evaluator.evaluate(
        callback=hf_inference(model_name),
        workers=WORKERS,
    )


In [ ]:
# Save every model's report: Results/<model_name>/performance.png + metrics.json,
# plus Results/comparison.png + comparison.json when several models ran.
summary = report_results(reports)
